In [3]:
import numpy as np
import json
import gzip
from bs4 import BeautifulSoup
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

X_vec = np.load("html_lm_vectors.npy")
splits = np.load("splits.npy")
categories = np.load("categories.npy")

train_mask = (splits == "train")
test_mask = (splits == "test")

X_train, y_train = X_vec[train_mask], categories[train_mask]
X_test, y_test = X_vec[test_mask], categories[test_mask]

clf_htmllm = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1))
clf_htmllm.fit(X_train, y_train)

y_pred_htmllm = clf_htmllm.predict(X_test)
print("Přesnost HTML-LM modelu:", accuracy_score(y_test, y_pred_htmllm))
print(classification_report(y_test, y_pred_htmllm))

with gzip.open("curlie_cs.jsonl.gz", "rt", encoding="utf-8") as f:
    pages = [json.loads(line) for line in f]

texty = [BeautifulSoup(p["html"], "html.parser").get_text(" ") for p in pages]
texty_train = [texty[i] for i in range(len(pages)) if pages[i]["split"] == "train"]
texty_test = [texty[i] for i in range(len(pages)) if pages[i]["split"] == "test"]

y_train_tfidf = np.array([p["category"] for p in pages if p["split"] == "train"])
y_test_tfidf = np.array([p["category"] for p in pages if p["split"] == "test"])

tfidf = TfidfVectorizer(min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(texty_train)
X_test_tfidf = tfidf.transform(texty_test)

clf_tfidf = LogisticRegression(max_iter=3000, C=10)
clf_tfidf.fit(X_train_tfidf, y_train_tfidf)

y_pred_tfidf = clf_tfidf.predict(X_test_tfidf)
print("Přesnost TF-IDF modelu:", accuracy_score(y_test_tfidf, y_pred_tfidf))
print(classification_report(y_test_tfidf, y_pred_tfidf))

Přesnost HTML-LM modelu: 0.7548672566371681
                precision    recall  f1-score   support

          Arts       0.78      0.77      0.77       116
      Business       0.82      0.82      0.82       263
     Computers       0.79      0.74      0.76        87
         Games       0.69      0.43      0.53        21
        Health       0.78      0.85      0.82        62
          Home       0.64      0.75      0.69        12
Kids_and_Teens       0.66      0.71      0.69        94
          News       0.71      0.71      0.71         7
    Recreation       0.77      0.79      0.78       147
     Reference       0.72      0.61      0.66        46
       Science       0.59      0.47      0.53        40
      Shopping       0.75      0.73      0.74        88
       Society       0.64      0.74      0.69        73
        Sports       0.76      0.81      0.78        74

      accuracy                           0.75      1130
     macro avg       0.72      0.71      0.71      1130
  